# Speculative Decoding

**Tags:** optimization, inference
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/speculative-decoding.md

## TL;DR

Parallelize LLM generation using smaller draft model: draft predicts k tokens fast (2-5ms), verify all k with large model in parallel (10ms). Accept tokens until divergence found. Speedup: 2-4x with identical quality. Trade-off: requires draft model, 10-20% extra compute.

## Core Intuition

Standard generation is sequential: generate token 1, then token 2, then token 3... Each takes 10ms. Speculative decoding "speculates" with a fast draft model: generate tokens 1-4 in 2-5ms, then verify all 4 simultaneously with large model in 10ms. If all 4 match, accept them all. If divergence at token 3, accept 1-2, reject 3-4, continue. Result: generate ~4 tokens in ~10ms vs ~40ms (4x faster).

## How It Works

**Standard Autoregressive Generation (Sequential):**
```
Input: "The future of AI"
Sample token 1 from large model: "is" (10ms)
Input: "The future of AI is"
Sample token 2: "bright" (10ms)
Input: "The future of AI is bright"
Sample token 3: "and" (10ms)
...

Total for 100 tokens: 100 × 10ms = 1000ms (slow!)
```

**Speculative Decoding (Parallel Verification):**
```
Input: "The future of AI"

Iteration 1:
  Draft phase: fast model generates k=4 tokens (2ms)
    "is bright and" + "exciting"
  Verify phase: large model processes all 4 tokens (10ms)
    logits for positions 1, 2, 3, 4
  
  Check acceptance (greedy comparison):
    Token 1: draft="is" vs large sample="is" → match ✓
    Token 2: draft="bright" vs large sample="bright" → match ✓
    Token 3: draft="and" vs large sample="and" → match ✓
    Token 4: draft="exciting" vs large sample="exciting" → match ✓
  
  Accept all 4 tokens (accept_count=4)
  Large model generates 1 additional token from position 5: "in"
  
  Time: 2ms (draft) + 10ms (verify) = 12ms for 5 tokens

Iteration 2:
  Input: "The future of AI is bright and exciting in"
  (repeat...)

Total for 100 tokens: ~100 / 5 × 12ms = 240ms (5x faster!)
```

**Algorithm (with probability sampling):**

```python
while not_done:
    # Draft phase (fast): generate k tokens
    draft_tokens = []
    draft_logits = []
    
    for i in range(k):
        logits = draft_model(sequence)
        token = sample(logits)  # or greedy
        draft_tokens.append(token)
        draft_logits.append(logits)
        sequence.append(token)
    
    # Verify phase (large model): forward pass on k tokens
    large_logits = large_model(sequence[-(k+1):])  # forward only on new context
    
    # Acceptance check (sampling-based)
    # Sample from large_logits and compare with draft_tokens
    accept_count = 0
    for i in range(k):
        large_logit = large_logits[i]
        draft_logit = draft_logits[i]
        
        # Probability of draft token under large model
        large_prob = softmax(large_logit)[draft_tokens[i]]
        
        # Rejection probability: min(1, large_prob / draft_prob)
        draft_prob = softmax(draft_logit)[draft_tokens[i]]
        reject_prob = 1 - min(1, large_prob / draft_prob)
        
        if reject_prob > random():
            # Rejected
            break
        else:
            # Accepted
            accept_count += 1
    
    # At divergence or end of draft tokens:
    # Large model samples 1 new token
    if accept_count < k:
        # Resample token at position accept_count with corrected distribution
        # (to correct for rejection bias)
        new_token = sample(corrected_logits)
    else:
        # All accepted, sample next
        new_token = sample(large_logits[k])
    
    sequence.append(new_token)
    
    if sequence ends or reaches max_length:
        break
```

### Workflow Diagram

```mermaid
graph LR
    A["Input"] --> B["Speculative Decoding Process"]
    B --> C["Output"]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#e8f5e9
```

**Note:** This is a basic workflow template. Review and customize based on specific concept.

## Key Properties & Trade-offs

/ Trade-offs

| Metric | Standard | Speculative (k=5) | Notes |
|--------|----------|------------------|-------|
| Latency | 1000ms (100 tok) | 240ms (100 tok) | 4-5x faster |
| Quality | 100% | 100% | Identical (correct sampling) |
| Compute | 100 forward passes | 120 forward passes | +20% compute (small overhead) |
| Memory | Model memory | Model + draft memory | +10-20% RAM (draft model smaller) |
| Complexity | Simple | Complex (sampling logic) | More engineering |
| Throughput | Same | Same | Latency improvement, not throughput |

**When Speculative Decoding Works Well:**

```
Good scenarios:
  - Draft model well-matched to large model
    (e.g., quantized large model + full-precision as draft)
  - Long generation lengths (100+ tokens)
  - High-quality draft model (agreement rate >80%)

Poor scenarios:
  - Draft quality far from large model
    → high rejection rate → no speedup
  - Very short generations (1-5 tokens)
    → overhead of parallel verification not worth it
  - Large model already fast
    → optimizing generation not the bottleneck
```

**Speedup Analysis:**

```
Assumption: draft model 10x faster than large model

Scenario 1 (perfect agreement k=5):
  Standard: 100 tokens × T_large = 100T
  Speculative: (100/5) × (T_draft + T_large) = 20 × (T_large/10 + T_large) = 22T
  Speedup: 100T / 22T ≈ 4.5x

Scenario 2 (80% agreement, k=5):
  Rejection rate: 20%
  Avg tokens per iteration: 0.8 × 5 + 1 = 5
  Speculative: (100/5) × 1.1 × (T_draft + T_large) ≈ 24T
  Speedup: 100T / 24T ≈ 4.2x

Scenario 3 (50% agreement, k=5):
  Avg tokens per iteration: 0.5 × 5 + 1 = 3.5
  Speculative: (100/3.5) × 1.15 × (T_draft + T_large) ≈ 40T
  Speedup: 100T / 40T = 2.5x
```

### Code Implementation

```python
# Key imports for Speculative Decoding
import numpy as np
import torch
from typing import Any

# Speculative Decoding example implementation
class SpeculativeDecoding:
    """
    Speculative Decoding implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts

```mermaid
graph TD
    A["Speculative Decoding"]
    A -->|used with| D["Attention Optimization"]
    
    style A fill:#fff3e0
```

### Common Interview Questions

**Q: What is Speculative Decoding used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Speculative Decoding?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Speculative Decoding compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Speculative Decoding?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Speculative Decoding?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/speculative-decoding.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]